# CONAB Data Explorer — ABIOVE LULC 2026
**Purpose:** Understand every new CONAB file before integrating into the pipeline.  
**Output:** `processed/conab_cafe_uf.csv` and `processed/conab_milho_split_uf.csv`  
**Year range:** 2008–2024

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT     = Path('..').resolve()
CONAB    = ROOT / 'Dados_CONAB'
ILUC     = ROOT / 'ILUC_NIPE'
PAM_FILE = ILUC / '05_Agro_Subdivisions' / 'PAM_RGINT_COMPLETO.csv'
PROC     = Path('.') / 'processed'
PROC.mkdir(exist_ok=True)

YEARS = list(range(2008, 2025))
print('Project root:', ROOT)
print('CONAB files:'); [print(' ', f.name) for f in sorted(CONAB.iterdir()) if not f.suffix == '.zip']

## 1. PAM RGINT — check if café is already there

In [ ]:
pam = pd.read_csv(PAM_FILE, dtype={'CD_RGINT': str})
pam.columns = pam.columns.str.strip()
print('PAM columns:', pam.columns.tolist())
print('\nCulturas únicas na PAM:')
print(sorted(pam['cultura'].unique()))

## 2. SerieHistoricaGraos — soja, milho, algodão, arroz por UF

In [ ]:
graos = pd.read_csv(CONAB / 'SerieHistoricaGraos.txt', sep=';', decimal=',', encoding='utf-8')
graos.columns = graos.columns.str.strip()
print('Colunas:', graos.columns.tolist())
print('\nProdutos únicos:', sorted(graos['produto'].unique()))
print('\nAnos:', sorted(graos['ano_agricola'].unique())[:10], '...')
print('\nShape:', graos.shape)

In [ ]:
# Soja e milho — cobertura por UF
for prod in ['SOJA', 'MILHO']:
    sub = graos[graos['produto'].str.upper().str.contains(prod, na=False)]
    print(f'{prod}: {len(sub)} linhas | UFs: {sorted(sub["uf"].unique())}')

## 3. LevantamentoGraos — split 1ª/2ª safra para milho (KEY)

In [ ]:
lev = pd.read_csv(CONAB / 'LevantamentoGraos.txt', sep=';', decimal=',', encoding='utf-8')
lev.columns = lev.columns.str.strip()
print('Colunas:', lev.columns.tolist())
print('\nSafras únicas:', sorted(lev['safra'].dropna().unique()))
print('\nProdutos únicos:', sorted(lev['produto'].unique()))
print('\nAnos agrícolas:', sorted(lev['ano_agricola'].unique()))

In [ ]:
# Filtrar milho com split 1a/2a
milho_mask = lev['produto'].str.upper().str.contains('MILHO', na=False)
milho = lev[milho_mask].copy()
print('Safras de milho:', sorted(milho['safra'].dropna().unique()))
print('Levantamentos (rounds):', sorted(milho['id_levantamento'].dropna().unique()))
milho.groupby(['ano_agricola', 'safra'])['area_plantada_mil_ha'].sum().unstack().tail(10)

In [ ]:
# Usar último levantamento por ano/UF/safra
milho['id_lev_num'] = pd.to_numeric(milho['id_levantamento'], errors='coerce')
milho_final = (
    milho.sort_values('id_lev_num')
    .groupby(['ano_agricola', 'uf', 'safra'], as_index=False)
    .last()
)

# Parse ano_agricola: '2017/18' → 2017
milho_final['year'] = milho_final['ano_agricola'].astype(str).str[:4].astype(int)
milho_final = milho_final[milho_final['year'].between(2008, 2024)]

# Pivot 1a vs 2a
milho_pivot = milho_final.pivot_table(
    index=['year', 'uf'], columns='safra', values='area_plantada_mil_ha', aggfunc='sum'
).reset_index()
print('Colunas após pivot:', milho_pivot.columns.tolist())
milho_pivot.head()

In [ ]:
# Compute pct_2a = 2a / (1a + 2a) per UF/year
# Column names may vary — find them
cols = milho_pivot.columns.tolist()
col_1a = next((c for c in cols if '1' in str(c) and 'SAFRA' in str(c).upper()), None)
col_2a = next((c for c in cols if '2' in str(c) and 'SAFRA' in str(c).upper()), None)
print(f'Coluna 1a safra: {col_1a}  |  Coluna 2a safra: {col_2a}')

if col_1a and col_2a:
    milho_pivot['a1'] = pd.to_numeric(milho_pivot[col_1a], errors='coerce').fillna(0)
    milho_pivot['a2'] = pd.to_numeric(milho_pivot[col_2a], errors='coerce').fillna(0)
    milho_pivot['total'] = milho_pivot['a1'] + milho_pivot['a2']
    milho_pivot['pct_2a'] = np.where(milho_pivot['total'] > 0,
                                     milho_pivot['a2'] / milho_pivot['total'], np.nan)
    milho_split = milho_pivot[['year', 'uf', 'pct_2a']].dropna(subset=['pct_2a'])
    milho_split.to_csv(PROC / 'conab_milho_split_uf.csv', index=False, encoding='utf-8')
    print(f'Saved conab_milho_split_uf.csv — {len(milho_split)} rows')
    print('Coverage por ano:')
    print(milho_split.groupby('year')['uf'].count())
else:
    print('AVISO: colunas 1a/2a nao encontradas. Verifique os valores de safra acima.')

## 4. SerieHistoricaCafe — café por UF (para classe 1 - Culturas perenes)

In [ ]:
cafe = pd.read_csv(CONAB / 'SerieHistoricaCafe.txt', sep=';', decimal=',', encoding='utf-8')
cafe.columns = cafe.columns.str.strip()
print('Colunas:', cafe.columns.tolist())
print('\nProdutos:', sorted(cafe['produto'].unique()))
print('id_produto únicos:', sorted(cafe['id_produto'].unique()))
print('\nAnos:', sorted(cafe['ano_agricola'].astype(str).unique())[:10], '...')
cafe.head(3)

In [ ]:
# Agregar: soma das duas variedades (7090 + 7498) por UF/ano
cafe['year'] = pd.to_numeric(cafe['ano_agricola'], errors='coerce').astype('Int64')
cafe['area_ha'] = pd.to_numeric(
    cafe['area_plantada_mil_ha'].astype(str).str.replace(',', '.'), errors='coerce'
) * 1000   # mil ha → ha

cafe_uf = (
    cafe[(cafe['year'] >= 2008) & (cafe['year'] <= 2024)]
    .groupby(['year', 'uf'], as_index=False)['area_ha']
    .sum()
)
print('Shape:', cafe_uf.shape)
print('Anos cobertos:', sorted(cafe_uf['year'].unique()))
print('UFs cobertos:', sorted(cafe_uf['uf'].dropna().unique()))
cafe_uf.head(5)

In [ ]:
cafe_uf.to_csv(PROC / 'conab_cafe_uf.csv', index=False, encoding='utf-8')
print(f'Saved conab_cafe_uf.csv — {len(cafe_uf)} rows')

## 5. SerieHistoricaCana — validação da classe 5

In [ ]:
cana = pd.read_csv(CONAB / 'SerieHistoricaCana.txt', sep=';', decimal=',', encoding='utf-8')
cana.columns = cana.columns.str.strip()
print('Colunas:', cana.columns.tolist())
print('Anos:', sorted(cana['ano_agricola'].astype(str).unique()))
print('UFs:', sorted(cana['uf'].unique()))

## 6. Tabela 5457 (IBGE mesoregions) — estrutura apenas

In [ ]:
import openpyxl
wb = openpyxl.load_workbook(CONAB / 'tabela5457.xlsx', read_only=True, data_only=True)
ws = wb.active
print(f'Planilha: {ws.title} | Dimensões: {ws.dimensions}')
rows = list(ws.iter_rows(max_row=4, values_only=True))
print('Primeiras 4 linhas (primeiras 8 cols):')
for r in rows:
    print(list(r[:8]))
wb.close()
# Conclusão: 1227 colunas, pivot IBGE — defer para notebook separado

## 7. ZIP files — conteúdo de um exemplo

In [ ]:
import zipfile
sample_zip = next(CONAB.glob('MT_SOJA_2021.zip'), None) or next(CONAB.glob('*.zip'))
print('Arquivo:', sample_zip.name)
with zipfile.ZipFile(sample_zip) as zf:
    print('Conteúdo:', zf.namelist())
# Shapefiles CONAB por estado/cultura/safra — join espacial necessário, adiado

## 8. Resumo — o que entra no pipeline

| Arquivo | Uso |
|---|---|
| `PAM_RGINT_COMPLETO.csv` | Nível RGINT — soja, milho, cana (já integrado) + café (se existir) |
| `SerieHistoricaCafe.txt` | UF → RGINT via peso PAM → **nova fonte para classe 1** |
| `LevantamentoGraos.txt` | Split milho 1ª/2ª safra por UF → **corrige classes 3+4** |
| `SerieHistoricaCana.txt` | Validação visual UF, classe 5 |
| `SerieHistoricaGraos.txt` | Referência UF para soja/milho/algodão |
| `tabela5457.xlsx` | Defer — pivot complexo, mesorregião ≠ RGINT |
| `*.zip` (shapefiles) | Defer — join espacial, baixo ganho marginal |